# Diabetic Retinopathy — Training Notebook (Colab)

**Workflow:**
1. Mount Google Drive
2. Set `STRATEGY` (and `BACKBONE` if fine-tuning)
3. Run all cells — the notebook trains the model, saves a named checkpoint to Drive, and generates a ready-to-submit ZIP.

**Drive layout expected:**
```
MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/
  data/          ← images + train/val/test CSVs
  src/           ← Python modules (copy of repo src/)
  models/
    custom/      ← one .pth per completed run, e.g. custom_auc0.742_20260416_143022.pth
    fine_tune/   ← e.g. fine_tune_alexnet_auc0.821_20260416_150311.pth
  results/
    custom_results/      ← one .csv per completed run
    fine_tune_results/   ← one .csv per completed run
    Submissions/         ← codabench_submission.zip (always up to date)
```

**Naming convention:** `{strategy}[_{backbone}]_auc{val_auc}_{YYYYMMDD_HHMMSS}`

## 1. Mount Drive & set up paths

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import sys
import os

DRIVE_BASE   = '/content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy'
DATA_DIR     = os.path.join(DRIVE_BASE, 'data')
# Updated to the correct path
SRC_DIR      = os.path.join(DRIVE_BASE, 'src')
MODELS_DIR   = os.path.join(DRIVE_BASE, 'models')
RESULTS_DIR  = os.path.join(DRIVE_BASE, 'results')

# Make both the base project directory and the src directory importable
for path in [DRIVE_BASE, SRC_DIR]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Create output directories
for d in [
    os.path.join(MODELS_DIR,  'custom'),
    os.path.join(MODELS_DIR,  'fine_tune'),
    os.path.join(RESULTS_DIR, 'custom_results'),
    os.path.join(RESULTS_DIR, 'fine_tune_results'),
    os.path.join(RESULTS_DIR, 'Submissions'),
]:
    os.makedirs(d, exist_ok=True)

print(f'Paths updated. SRC_DIR set to: {SRC_DIR}')

Paths updated. SRC_DIR set to: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src


### 1.1 Speed up Training: Copy Data to Local Storage
We copy the dataset from Drive to the local Colab runtime to avoid slow network I/O during training.

In [25]:
import shutil
import os

# Define local path
LOCAL_DATA_DIR = '/content/data'

if not os.path.exists(LOCAL_DATA_DIR):
    print("Copying data to local storage...")
    # Assuming images are in a folder or zip. Adjust if you have a .zip file specifically.
    # If it's a folder of images:
    shutil.copytree(DATA_DIR, LOCAL_DATA_DIR)
    print("Data copied to local storage.")

# Update DATA_DIR to point to local disk
DATA_DIR = LOCAL_DATA_DIR
print(f"Updated DATA_DIR to: {DATA_DIR}")

Updated DATA_DIR to: /content/data


In [26]:
import os
# Diagnostic to find any .py files in the new SRC_DIR
print(f"Buscando archivos de código en: {SRC_DIR}")

if os.path.exists(SRC_DIR):
    found_code = False
    for root, dirs, files in os.walk(SRC_DIR):
        for file in files:
            if file.endswith('.py'):
                print(f"Encontrado: {os.path.join(root, file)}")
                found_code = True

    if not found_code:
        print("No se encontraron archivos .py en SRC_DIR. ¿Estás seguro de que esta es la carpeta correcta?")
else:
    print(f"La ruta SRC_DIR no es válida: {SRC_DIR}")

Buscando archivos de código en: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/__init__.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/data/transforms.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/data/dataset.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/data/__init__.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/model/__init__.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/model/custom_net.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/model/fine_tune_net.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinop

## 2. Install dependencies *(run once if needed)*

In [27]:
# Uncomment on first run in a fresh Colab session
# !pip install -q scikit-image opencv-python-headless scikit-learn

## 3. Imports

In [28]:
import random
import shutil
from datetime import datetime
import sys
import os

# Force path priority
SRC_DIR = '/content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src'
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Clear potential namespace conflict if 'data' was already imported elsewhere
if 'data' in sys.modules:
    del sys.modules['data']

import numpy as np
import numpy.random as npr
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

# Import the sub-packages directly
try:
    from data.dataset    import RetinopathyDataset
    from data.transforms import get_train_transforms, get_val_transforms
    from model.custom_net    import CustomNet
    from model.ensemble_net  import EnsembleNet
    from model.fine_tune_net import build_fine_tune_model
    from training.trainer    import train_model
    from training            import config
    from evaluation.submission import (
        test_model, save_strategy_results, generate_submission
    )
    print('All imports OK.')
except ImportError as e:
    print(f'Import failed: {e}')
    print(f'Current sys.path: {sys.path[:3]}')
    if os.path.exists(os.path.join(SRC_DIR, 'data', 'dataset.py')):
        print('Verification: data/dataset.py exists in SRC_DIR.')
    else:
        print('Verification: data/dataset.py NOT found in SRC_DIR.')


All imports OK.


## 4. Configuration

**Edit this cell** before each training run.

In [29]:
import os

print("Buscando directorios importantes...")
drive_path = '/content/drive/MyDrive/'

# Buscar carpeta src
for root, dirs, files in os.walk(drive_path):
    if 'src' in dirs:
        print(f"Posible carpeta src encontrada: {os.path.join(root, 'src')}")
    if 'AutomaticDiagnosisForDiabeticRetinopathy' in dirs:
        print(f"Posible carpeta del proyecto encontrada: {os.path.join(root, 'AutomaticDiagnosisForDiabeticRetinopathy')}")

print("\n--- Configuración original ---")
# ── Strategy ──────────────────────────────────────────────────────────────────
STRATEGY = 'custom'        # 'custom'  →  train CustomNet from scratch
                            # 'fine_tune' →  fine-tune a pretrained backbone

# ── Fine-tune backbone (ignored when STRATEGY='custom') ───────────────────────
BACKBONE = 'alexnet'        # 'alexnet' | 'resnet18' | 'resnet50' | 'vgg16' | 'efficientnet_b0'

BATCH_SIZE   = config.batch_size
NUM_EPOCHS   = config.num_epochs
LR           = config.learning_rate
LR_STEP_SIZE = config.lr_step_size
LR_GAMMA     = config.lr_gamma
MOMENTUM     = config.momentum

# Set to a small integer (e.g. 200) for fast debug runs; 0 = full dataset
MAX_TRAIN_SIZE = 0

# ── Temp checkpoint (overwritten during training, deleted after run) ──────────
TEMP_SAVE_PATH = os.path.join(MODELS_DIR, STRATEGY, '_temp_best.pth')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')
print(f'Strategy : {STRATEGY}')
if STRATEGY == 'fine_tune':
    print(f'Backbone : {BACKBONE}')
print(f'Epochs   : {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}')


Buscando directorios importantes...
Posible carpeta del proyecto encontrada: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy
Posible carpeta src encontrada: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src

--- Configuración original ---
Device   : cuda
Strategy : custom
Epochs   : 50  |  Batch: 32  |  LR: 0.01


## 5. Data loading

In [30]:
import random
import numpy.random as npr
import torch
import os
from torch.utils.data import DataLoader

# Ensure source modules and config values are available
from data.dataset import RetinopathyDataset
from data.transforms import get_train_transforms, get_val_transforms
from training import config

# Use local data for speed
DATA_DIR = '/content/data'
MAX_TRAIN_SIZE = 0
BATCH_SIZE = config.batch_size

random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.enabled = False

train_transform = get_train_transforms(img_size=config.img_height)
val_transform   = get_val_transforms(img_size=config.img_height)

train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'train.csv'),
    root_dir=DATA_DIR,
    transform=train_transform,
    maxSize=MAX_TRAIN_SIZE,
)
val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'val.csv'),
    root_dir=DATA_DIR,
    transform=val_transform,
)
test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'test.csv'),
    root_dir=DATA_DIR,
    transform=val_transform,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=256,         shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256,         shuffle=False, num_workers=2, pin_memory=True)

print(f'Using LOCAL data at: {DATA_DIR}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

Using LOCAL data at: /content/data
Train: 2000 | Val: 500 | Test: 1000


## 6. Build model

In [31]:
if STRATEGY == 'custom':
    # ── EnsembleNet: DenseNetSmall (green ch) + ResNet-9 (RGB) ───────────────
    model     = EnsembleNet().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config.ensemble_lr,
        weight_decay=config.ensemble_weight_decay,
    )
    scheduler = lr_scheduler.StepLR(
        optimizer,
        step_size=config.ensemble_lr_step_size,
        gamma=config.ensemble_lr_gamma,
    )
    NUM_EPOCHS = config.ensemble_num_epochs
    BATCH_SIZE = config.ensemble_batch_size

else:
    # ── Fine-tune: pretrained torchvision backbone ────────────────────────────
    model     = build_fine_tune_model(backbone=BACKBONE).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
    scheduler = lr_scheduler.StepLR(
        optimizer,
        step_size=LR_STEP_SIZE,
        gamma=LR_GAMMA,
    )

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model       : {model.__class__.__name__}')
print(f'Criterion   : {criterion.__class__.__name__}')
print(f'Optimizer   : {optimizer.__class__.__name__}')
print(f'Trainable parameters: {n_params:,}')
print(f'Epochs      : {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {optimizer.param_groups[0]["lr"]:.2e}')


Trainable parameters: 845,337


## 7. Train

The running best is kept at `_temp_best.pth` during training.  
Once done, it is renamed to a permanent file: `{strategy}[_{backbone}]_auc{auc}_{timestamp}.pth`.

In [32]:
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

dataloaders   = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

model, best_auc = train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    dataloaders=dataloaders,
    dataset_sizes=dataset_sizes,
    device=device,
    num_epochs=NUM_EPOCHS,
    temp_save_path=TEMP_SAVE_PATH,
)

# ── Build permanent name and move temp checkpoint ────────────────────────────
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backbone_tag = f'_{BACKBONE}' if STRATEGY == 'fine_tune' else ''
RUN_NAME = f'{STRATEGY}{backbone_tag}_auc{best_auc:.4f}_{timestamp}'

final_model_path = os.path.join(MODELS_DIR, STRATEGY, f'{RUN_NAME}.pth')
if os.path.exists(TEMP_SAVE_PATH):
    shutil.move(TEMP_SAVE_PATH, final_model_path)
    print(f'Model saved -> {final_model_path}')
else:
    # No improvement at all during training — save current weights anyway
    torch.save(model.state_dict(), final_model_path)
    print(f'Model saved (no improvement during training) -> {final_model_path}')

print(f'Run name: {RUN_NAME}')

Epoch 0/49
----------
  train  loss: 0.6170  AUC: 0.5035
  val    loss: 0.5678  AUC: 0.5492
  -> New best (AUC=0.5492) saved to temp checkpoint

Epoch 1/49
----------


KeyboardInterrupt: 

## 8. Generate submission

- Runs inference on the test set.
- Saves scores as `{run_name}.csv` in `results/{strategy}_results/`.
- Looks for the other strategy's most recent CSV; uses **random scores** if none found.
- Creates `results/Submissions/codabench_submission.zip`.

In [ ]:
# Run test inference
outputs = test_model(model, test_loader, device)
print(f'Test outputs shape: {outputs.shape}')   # expected (1000, 1)

# Save this run's scores with the same descriptive name as the model
current_csv_path = save_strategy_results(outputs, STRATEGY, RESULTS_DIR, RUN_NAME)

# Build submission ZIP (combines both strategies)
zip_path = generate_submission(
    current_strategy=STRATEGY,
    results_dir=RESULTS_DIR,
    current_csv_path=current_csv_path,
    n_test=len(test_dataset),
)
print(f'\nDone! Upload to Codabench: {zip_path}')